In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from collections import defaultdict
import numpy as np
import pytz
import re

In [2]:

# Read the list of filenames from the configuration file
with open('file_list.txt', 'r', encoding='utf-8') as config_file:
    file_names = config_file.read().splitlines()

# Regex pattern to match the data format
pattern = r'\[(.*?)\] (.*?): (.*)'

# Initialize an empty list to store parsed data
datalist = []
stream_count = 0
# Iterate over each specified file
for file in file_names:
    full_path = "data\\"+file
    with open(full_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines:
            match = re.match(pattern, line)
            if match:
                date, user, message = match.groups()
                datalist.append([date, user, message,stream_count])
    stream_count = stream_count + 1

# Create a DataFrame from the parsed data
data = pd.DataFrame(datalist, columns=["date", "user", "message","stream"])

In [3]:
data['date'] = pd.to_datetime(data['date'])

In [9]:

import pandas as pd
import requests
import time
from urllib.parse import quote

from collections import defaultdict

# ==========================================
# SETTINGS
# ==========================================

TOP_N = 1000
DELAY = 0.2  # Delay between requests
BASE_URL = "https://logs.zonian.dev/namehistory/login:"

# ==========================================
# 1. GET TOP 1000 CHATTERS
# ==========================================

top_users = (
    data["user"]
    .dropna()
    .astype(str)
    .value_counts()
    .head(TOP_N)
)

print(f"Checking name history for {len(top_users)} users...")

# ==========================================
# 2. FETCH NAME HISTORY
# ==========================================

session = requests.Session()

replacements = {}
errors = []
history_results = []

for i, username in enumerate(top_users.index, 1):

    try:
        url = BASE_URL + quote(username, safe="")

        response = session.get(url, timeout=15)

        if response.status_code == 404:
            continue

        response.raise_for_status()

        history = response.json()

        if not isinstance(history, list) or not history:
            continue

        # Remove empty usernames
        history = [
            entry for entry in history
            if entry.get("user_login")
        ]

        if not history:
            continue

        # Find the newest username based on last seen timestamp
        newest = max(
            history,
            key=lambda x: x["last_timestamp"]
        )["user_login"]

        # Map every previous username to the newest one
        for entry in history:

            old_username = entry["user_login"]

            if old_username == newest:
                continue

            # Detect conflicting mappings
            if old_username in replacements:
                if replacements[old_username] != newest:
                    errors.append({
                        "username": old_username,
                        "error": f"Conflicting mapping: "
                                 f"{replacements[old_username]} / {newest}"
                    })
                    continue

            replacements[old_username] = newest

        history_results.append({
            "queried_username": username,
            "newest_username": newest,
            "names_found": len(history),
            "history": ", ".join(
                entry["user_login"] for entry in history
            )
        })

        if len(history) > 1:
            print(f"[{i}/{len(top_users)}] {username} -> {newest}")

        if i % 50 == 0:
            print(f"Progress: {i}/{len(top_users)}")

    except Exception as e:
        errors.append({
            "username": username,
            "error": str(e)
        })

        print(f"Error checking {username}: {e}")

    time.sleep(DELAY)

# ==========================================
# 3. GENERATE PYTHON REPLACEMENT CODE
# ==========================================

print("\nGenerating replacement code...")

# Use one dictionary replacement to avoid chained replacements
# accidentally changing usernames more than once.

code_lines = [
    "# Automatically generated Twitch username replacements",
    "",
    "data['user'] = data['user'].replace({"
]

for old, new in sorted(replacements.items()):
    code_lines.append(f"    {old!r}: {new!r},")

code_lines.append("})")

with open("username_replacements.py", "w", encoding="utf-8") as f:
    f.write("\n".join(code_lines))

# ==========================================
# 4. SAVE ADDITIONAL RESULTS
# ==========================================

pd.DataFrame(history_results).to_csv(
    "username_history.csv",
    index=False
)

pd.DataFrame(errors).to_csv(
    "username_errors.csv",
    index=False
)

# ==========================================
# 5. SUMMARY
# ==========================================

print("\n" + "=" * 50)
print("FINISHED!")
print("=" * 50)

print(f"Users checked:       {len(top_users)}")
print(f"Histories retrieved: {len(history_results)}")
print(f"Replacements found:  {len(replacements)}")
print(f"Errors:              {len(errors)}")

print("\nGenerated files:")
print("- username_replacements.py")
print("- username_history.csv")
print("- username_errors.csv")


Checking name history for 1000 users...
[1/1000] JBIN2036 -> jbin2036
[7/1000] FilipStayout -> filipstayout
[11/1000] HALP____ -> halp____
[12/1000] nishad13 -> nishad13
[13/1000] Typhu25 -> typhu25
[16/1000] rautsi__ -> rautsi__
[17/1000] Martin_Gales -> martin_gales
[18/1000] Wirelesss_ -> wirelesss_
[21/1000] KrieSeaX -> krieseax
[24/1000] CrazeE420xd -> crazee420xd
[25/1000] elluiti -> elluiti
[27/1000] klimzaa -> klimzaa
[30/1000] Muuskie -> muuskie
[32/1000] banties_x -> banties_x
[33/1000] 1206paul_ -> 1206paul_
[35/1000] lajoss__ -> lajoss__
[39/1000] Foolsworn -> foolsworn
[44/1000] Georgie1471 -> georgie1471
[45/1000] Khukis -> khukis
[48/1000] haHAA_12_btw -> hahaa_12_btw
[49/1000] thebigdogjay -> thebigdogjay
[50/1000] IvanOnMyOwn -> ivanonmyown
Progress: 50/1000
[55/1000] phrank007 -> phrank007
[56/1000] 609__ -> 609__
[57/1000] 55Dash -> 55dash
[58/1000] amirmasoud_2018 -> amirmasoud_2018
[59/1000] Hueqi -> hueqi
[61/1000] Fossabot -> fossabot
[62/1000] catsspurr -> catss

In [10]:

import pandas as pd
import requests
import time
from urllib.parse import quote

BASE_URL = "https://logs.zonian.dev/namehistory/login:"

# Load previous errors
errors_df = pd.read_csv("username_errors.csv")
failed_users = errors_df["username"].dropna().unique()

# Load existing results
history_df = pd.read_csv("username_history.csv")

# Load existing replacement mappings
replacements = {}

for _, row in history_df.iterrows():
    history = str(row["history"]).split(", ")
    newest = row["newest_username"]

    for old in history:
        if old and old != newest:
            replacements[old] = newest

session = requests.Session()
new_errors = []
new_results = []

print(f"Retrying {len(failed_users)} failed users...")

for i, username in enumerate(failed_users, 1):
    try:
        url = BASE_URL + quote(str(username), safe="")
        response = session.get(url, timeout=20)
        response.raise_for_status()

        history = response.json()

        if not isinstance(history, list) or not history:
            raise ValueError("Empty or invalid history")

        history = [
            entry for entry in history
            if entry.get("user_login")
        ]

        if not history:
            raise ValueError("No usernames found")

        newest = max(
            history,
            key=lambda x: x["last_timestamp"]
        )["user_login"]

        for entry in history:
            old = entry["user_login"]

            if old != newest:
                if old not in replacements:
                    replacements[old] = newest
                elif replacements[old] != newest:
                    print(f"Conflict: {old} -> {newest}")

        new_results.append({
            "queried_username": username,
            "newest_username": newest,
            "names_found": len(history),
            "history": ", ".join(
                entry["user_login"] for entry in history
            )
        })

        print(f"[{i}/{len(failed_users)}] {username} -> {newest}")

    except Exception as e:
        new_errors.append({
            "username": username,
            "error": str(e)
        })
        print(f"Failed: {username}: {e}")

    time.sleep(0.3)

# Merge successful results
history_df = pd.concat(
    [history_df, pd.DataFrame(new_results)],
    ignore_index=True
).drop_duplicates(subset="queried_username", keep="last")

# Save updated history
history_df.to_csv("username_history.csv", index=False)

# Save remaining errors
pd.DataFrame(
    new_errors,
    columns=["username", "error"]
).to_csv("username_errors.csv", index=False)

# Regenerate replacement code
with open("username_replacements.py", "w", encoding="utf-8") as f:
    f.write("# Updated Twitch username replacements\n\n")
    f.write("data['user'] = data['user'].replace({\n")

    for old, new in sorted(replacements.items()):
        f.write(f"    {old!r}: {new!r},\n")

    f.write("})\n")

print("\nRetry complete!")
print(f"Successfully recovered: {len(new_results)}")
print(f"Still failing: {len(new_errors)}")
print(f"Total replacements: {len(replacements)}")


Retrying 129 failed users...
[1/129] HoneyKick -> honeykick
Failed: 알래스카해달: Empty or invalid history
[3/129] mangomuhid -> mangomuhid
[4/129] fluxen9 -> fluxen9
[5/129] fuzok4 -> fuzok4
[6/129] Roonss -> roonss
Failed: cukululu41: Empty or invalid history
[8/129] Vidke04 -> vidke04
[9/129] greekviewer1 -> greekviewer1
[10/129] tiberoo -> tiberoo
Failed: fyredoor4: Empty or invalid history
[12/129] TheDoctorFlash -> thedoctorflash
[13/129] irohthetealover -> irohthetealover
[14/129] ClarkKent_7 -> clarkkent_7
[15/129] GassAper -> gassaper
[16/129] Nightbot -> nightbot
[17/129] aransamus64 -> aransamus64
Failed: tiberiu0s: Empty or invalid history
[19/129] Ehas__ -> ehas__
[20/129] dTerreneitoRb -> dterreneitorb
[21/129] Drizzie3S -> drizzie3s
[22/129] stanny781 -> stanny781
[23/129] husvetteir -> husvetteir
[24/129] kepplerrino -> kepplerrino
[25/129] MrMcNoface -> mrmcnoface
[26/129] sl0ppy_g0bbler -> sl0ppy_g0bbler
[27/129] akshaykambe -> akshaykambe
Failed: smilodon_j: Empty or inval

In [5]:
data["user"] = data["user"].str.lower()

In [6]:
import re

# Get usernames present in your DataFrame
existing_users = set(data["user"].dropna().str.lower())

# Read replacement file
with open("username_replacements.py", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Keep only replacements where the old username exists in data
filtered_lines = []

for line in lines:
    match = re.match(r"\s*(['\"])(.*?)\1:\s*(['\"])(.*?)\3,", line)

    if match:
        old_username = match.group(2)

        if old_username.lower() in existing_users:
            filtered_lines.append(line)
    else:
        filtered_lines.append(line)

# Save filtered file
with open("username_replacements.py", "w", encoding="utf-8") as f:
    f.writelines(filtered_lines)

print("Replacement file cleaned!")

Replacement file cleaned!
